# 32-手写回测引擎

> 模块 4.2 回测与风控 | 从零构建 mini 回测系统，理解撮合逻辑与绩效计算

## 学习目标

- 理解回测引擎的五大核心组件：订单 → 撮合 → 账户 → 绩效 → 主循环
- 手写限价单和市价单的撮合逻辑（不依赖任何回测库）
- 实现精确的账户管理（现金、持仓、手续费）
- 计算夏普比、最大回撤、胜率、盈亏比等核心绩效指标
- 用模拟数据跑完整回测，对比买入持有基准

## 环境依赖

本 notebook 只用 `numpy`、`pandas`、`matplotlib`。不依赖 backtrader / vectorbt / zipline 等任何回测框架。

目标是理解引擎内部原理，而不是调包。学完这节课，你不会再对回测框架说「我不懂里面发生了什么」。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import List, Optional, Dict, Tuple
from enum import Enum
import warnings

warnings.filterwarnings("ignore")
plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## 1. 回测引擎架构全景

一个回测引擎本质上是**事件驱动的模拟器**：按时间推进，在每个时间点处理信号 → 生成订单 → 撮合成交 → 更新账户。

```text
                    ┌──────────────┐
                    │  信号/策略    │  ← 外部输入
                    └──────┬───────┘
                           ↓
┌──────────┐   ┌──────────┐   ┌──────────┐   ┌──────────┐
│  Order   │ → │ Matching │ → │ Account  │ → │Performance│
│ 订单系统  │   │ 撮合引擎  │   │ 账户管理  │   │ 绩效计算  │
└──────────┘   └──────────┘   └──────────┘   └──────────┘
     ↓              ↓              ↓              ↓
  限价/市价      价格检查       现金+持仓       Sharpe/回撤
  方向/数量      滑点模拟       手续费扣除      胜率/盈亏比
```

### 五个核心类

| 类 | 职责 | 关键逻辑 |
|------|------|------|
| **Order** | 表示一个订单 | 类型(limit/market)、方向、价格、数量、状态 |
| **MatchingEngine** | 撮合成交 | 限价单价格检查、市价单立即成交、滑点模拟 |
| **Account** | 管理资金与持仓 | 现金变动、持仓更新、手续费扣减、计算总市值 |
| **PerformanceCalculator** | 绩效评估 | 累计收益、年化Sharpe、最大回撤、胜率、盈亏比 |
| **BacktestEngine** | 主循环 | 逐日推进、信号→订单→撮合→账户→记录 |

接下来逐个实现。

## 2. 订单系统（Order）

订单是整个回测引擎的最小交易单位。限价单和市价单的区别：
- **市价单（Market Order）**：以当前市场价格立即成交
- **限价单（Limit Order）**：只有在市场价格达到或优于指定价格时才成交

真实回测中还有止损单、冰山单等，但这两个是最基础的。

In [ ]:
class OrderType(Enum):
    MARKET = "market"    # 市价单
    LIMIT = "limit"      # 限价单

class OrderSide(Enum):
    BUY = "buy"
    SELL = "sell"

class OrderStatus(Enum):
    PENDING = "pending"       # 待成交
    FILLED = "filled"         # 已成交
    CANCELLED = "cancelled"   # 已取消
    REJECTED = "rejected"     # 被拒绝

@dataclass
class Order:
    """订单"""
    order_id: int
    symbol: str
    order_type: OrderType
    side: OrderSide
    quantity: int          # 股数
    limit_price: Optional[float] = None   # 限价（仅限价单）
    status: OrderStatus = OrderStatus.PENDING
    filled_price: Optional[float] = None  # 实际成交价
    fee: float = 0.0
    timestamp: Optional[int] = None       # 下单时间（bar index）

    def __repr__(self):
        return (
            f"Order#{self.order_id} {self.side.value.upper()} "
            f"{self.quantity}股 @ {self.order_type.value} "
            f"[{self.status.value}]"
        )

    def is_valid(self) -> bool:
        """检查订单是否有效"""
        if self.order_type == OrderType.LIMIT and self.limit_price is None:
            return False
        if self.quantity <= 0:
            return False
        return True

# 演示
order1 = Order(1, "000001", OrderType.MARKET, OrderSide.BUY, 100)
order2 = Order(2, "000001", OrderType.LIMIT, OrderSide.BUY, 200, limit_price=10.5)
print(order1)
print(order2)
print(f"order1 valid: {order1.is_valid()}, order2 valid: {order2.is_valid()}")

## 3. 撮合引擎（MatchingEngine）

撮合引擎是回测系统的心脏。核心逻辑：

1. 接收订单 + 当前市场数据（价格、成交量）
2. 判断订单能否成交
   - 市价单：直接以当前价格成交
   - 限价买单：市场价 ≤ 限价才能成交
   - 限价卖单：市场价 ≥ 限价才能成交
3. 模拟滑点和手续费
4. 返回成交结果

> 注意：这里简化了——真实交易所的撮合要考虑对手盘、订单簿深度、排队顺序等。本节课聚焦于核心思想。

In [ ]:
@dataclass
class Fill:
    """成交记录"""
    order_id: int
    symbol: str
    side: OrderSide
    quantity: int
    price: float          # 成交价格
    fee: float            # 手续费
    timestamp: int        # 成交时间

@dataclass
class MarketData:
    """市场数据快照（每个 bar）"""
    timestamp: int
    open: float
    high: float
    low: float
    close: float
    volume: int

class MatchingEngine:
    """撮合引擎"""

    def __init__(self, slippage: float = 0.001, fee_rate: float = 0.0003):
        """
        Args:
            slippage: 滑点比例（默认 0.1%）
            fee_rate: 手续费率（默认 0.03%，万三）
        """
        self.slippage = slippage
        self.fee_rate = fee_rate
        self.fill_id = 0

    def match(self, order: Order, market_data: MarketData) -> Optional[Fill]:
        """尝试撮合订单

        Returns:
            Fill: 成交记录；None 表示未成交
        """
        if order.quantity <= 0:
            return None

        fill_price = self._determine_price(order, market_data)
        if fill_price is None:
            return None  # 限价单未达到条件

        # 应用滑点
        fill_price = self._apply_slippage(fill_price, order.side)

        # 计算手续费
        fee = fill_price * order.quantity * self.fee_rate

        self.fill_id += 1
        return Fill(
            order_id=order.order_id,
            symbol=order.symbol,
            side=order.side,
            quantity=order.quantity,
            price=fill_price,
            fee=fee,
            timestamp=market_data.timestamp,
        )

    def _determine_price(self, order: Order, md: MarketData) -> Optional[float]:
        """判断订单成交价格"""
        if order.order_type == OrderType.MARKET:
            return md.close  # 市价单以收盘价成交

        elif order.order_type == OrderType.LIMIT:
            return self._check_limit(order, md)

        return None

    def _check_limit(self, order: Order, md: MarketData) -> Optional[float]:
        """检查限价单能否成交

        限价买单：市场最低价 ≤ 限价 → 成交
        限价卖单：市场最高价 ≥ 限价 → 成交
        """
        if order.side == OrderSide.BUY:
            if md.low <= order.limit_price:
                # 以限价和 low 中较小者成交（对买家有利）
                return min(order.limit_price, md.close)
        else:
            if md.high >= order.limit_price:
                return max(order.limit_price, md.close)
        return None

    def _apply_slippage(self, price: float, side: OrderSide) -> float:
        """模拟滑点：买入价格略高，卖出价格略低"""
        if side == OrderSide.BUY:
            return price * (1 + self.slippage)
        else:
            return price * (1 - self.slippage)

# 演示撮合
md = MarketData(0, 10.0, 10.5, 9.8, 10.2, 100000)
engine = MatchingEngine(slippage=0.001, fee_rate=0.0003)

# 市价买单
market_buy = Order(1, "000001", OrderType.MARKET, OrderSide.BUY, 100)
fill = engine.match(market_buy, md)
print(f"市价买单: 成交价={fill.price:.4f}, 手续费={fill.fee:.4f}（滑点后）")

# 限价买单（限价 9.5，low=9.8，应该成交不了）
limit_buy = Order(2, "000001", OrderType.LIMIT, OrderSide.BUY, 200, limit_price=9.5)
fill2 = engine.match(limit_buy, md)
print(f"限价买单(9.5): {'成交' if fill2 else '未成交'}（low=9.8 > 限价9.5）")

# 限价买单（限价 10.5，low=9.8，应该成交）
limit_buy2 = Order(3, "000001", OrderType.LIMIT, OrderSide.BUY, 200, limit_price=10.5)
fill3 = engine.match(limit_buy2, md)
print(f"限价买单(10.5): 成交价={fill3.price:.4f}（限价10.5, close=10.2, 取min）")

## 4. 账户管理（Account）

Account 追踪回测中的资金状态：
- 现金余额
- 持仓（symbol → quantity）
- 每笔成交后更新现金和持仓
- 计算总市值 = 现金 + 持仓市值
- 检查是否有足够资金/持仓来执行订单

In [ ]:
@dataclass
class Account:
    """账户管理"""
    initial_cash: float = 1_000_000  # 初始资金 100万
    cash: float = None
    positions: Dict[str, int] = field(default_factory=dict)  # symbol -> quantity
    position_records: List[Dict] = field(default_factory=list)
    trade_records: List[Dict] = field(default_factory=list)

    def __post_init__(self):
        if self.cash is None:
            self.cash = self.initial_cash

    def can_afford(self, order: Order, price: float) -> bool:
        """检查是否有足够资金"""
        cost = price * order.quantity * (1 + 0.0003)  # 含预估手续费
        return self.cash >= cost

    def has_position(self, symbol: str, quantity: int) -> bool:
        """检查是否有足够持仓可卖"""
        return self.positions.get(symbol, 0) >= quantity

    def apply_fill(self, fill: Fill, current_price: float):
        """处理成交：更新现金和持仓"""
        symbol = fill.symbol

        if fill.side == OrderSide.BUY:
            cost = fill.price * fill.quantity + fill.fee
            self.cash -= cost
            self.positions[symbol] = self.positions.get(symbol, 0) + fill.quantity
        else:
            revenue = fill.price * fill.quantity - fill.fee
            self.cash += revenue
            self.positions[symbol] = self.positions.get(symbol, 0) - fill.quantity
            if self.positions[symbol] <= 0:
                del self.positions[symbol]

        # 记录交易
        self.trade_records.append({
            "timestamp": fill.timestamp,
            "symbol": fill.symbol,
            "side": fill.side.value,
            "quantity": fill.quantity,
            "price": fill.price,
            "fee": fill.fee,
        })

    def total_value(self, price_map: Dict[str, float]) -> float:
        """计算账户总市值"""
        position_value = sum(
            qty * price_map.get(sym, 0)
            for sym, qty in self.positions.items()
        )
        return self.cash + position_value

    def snapshot(self, timestamp: int, price_map: Dict[str, float]):
        """记录账户快照"""
        tv = self.total_value(price_map)
        self.position_records.append({
            "timestamp": timestamp,
            "cash": self.cash,
            "positions": dict(self.positions),
            "total_value": tv,
        })

# 演示
acct = Account(initial_cash=1000000)

# 买入
fill_buy = Fill(1, "000001", OrderSide.BUY, 1000, 10.0, 3.0, 0)
acct.apply_fill(fill_buy, 10.0)
print(f"买入后 — 现金: {acct.cash:.2f}, 持仓: {acct.positions}")

# 卖出
fill_sell = Fill(2, "000001", OrderSide.SELL, 500, 11.0, 1.65, 1)
acct.apply_fill(fill_sell, 11.0)
print(f"卖出后 — 现金: {acct.cash:.2f}, 持仓: {acct.positions}")

# 总市值
tv = acct.total_value({"000001": 11.0})
print(f"总市值: {tv:.2f} (现金{acct.cash:.2f} + 市值{500*11:.2f})")

## 5. 绩效计算（PerformanceCalculator）

有了账户快照序列，就可以计算标准绩效指标。这是回测结果的最终产出。

In [ ]:
class PerformanceCalculator:
    """绩效计算器"""

    @staticmethod
    def calculate(position_records: List[Dict], benchmark_returns: np.ndarray = None):
        """计算完整绩效指标"""
        navs = np.array([r["total_value"] for r in position_records])
        returns = np.diff(navs) / navs[:-1]

        if len(returns) == 0:
            return {"error": "无收益数据"}

        # 累计收益
        total_return = navs[-1] / navs[0] - 1

        # 年化收益率
        annual_return = (1 + total_return) ** (252 / len(returns)) - 1

        # 年化波动率
        annual_vol = np.std(returns) * np.sqrt(252)

        # Sharpe（假设无风险利率=0.02）
        rf_annual = 0.02
        sharpe = (annual_return - rf_annual) / annual_vol if annual_vol > 0 else 0

        # 最大回撤
        peak = np.maximum.accumulate(navs)
        drawdowns = (navs - peak) / peak
        max_dd = drawdowns.min()
        max_dd_idx = drawdowns.argmin()

        # Calmar 比率
        calmar = annual_return / abs(max_dd) if max_dd != 0 else 0

        # 胜率
        win_rate = (returns > 0).mean()

        # 盈亏比
        gains = returns[returns > 0]
        losses = returns[returns < 0]
        profit_loss_ratio = (
            gains.mean() / abs(losses.mean()) if len(losses) > 0 else float("inf")
        )

        # 年化换手率（简化：成交次数/总天数*252）
        # 这里先不计算，由外部传入

        result = {
            "累计收益": total_return,
            "年化收益": annual_return,
            "年化波动": annual_vol,
            "Sharpe": sharpe,
            "最大回撤": max_dd,
            "Calmar": calmar,
            "胜率": win_rate,
            "盈亏比": profit_loss_ratio,
            "nav": navs,
            "returns": returns,
            "drawdowns": drawdowns,
        }
        return result

    @staticmethod
    def summary(result: Dict) -> str:
        """格式化输出"""
        lines = []
        for k, v in result.items():
            if k in ("nav", "returns", "drawdowns"):
                continue
            if isinstance(v, float):
                if "率" in k or "回撤" in k:
                    lines.append(f"  {k}: {v:.2%}")
                else:
                    lines.append(f"  {k}: {v:.4f}")
        return "\n".join(lines)

## 6. 完整回测引擎

把所有组件串起来。主循环逻辑：

```text
for each bar (day):
    1. 策略生成信号
    2. 信号 → Order
    3. MatchingEngine.match(order, market_data)
    4. 如果成交 → Account.apply_fill(fill)
    5. Account.snapshot() 记录净值
end
performance = PerformanceCalculator.calculate(records)
```

In [ ]:
class BacktestEngine:
    """回测引擎主循环"""

    def __init__(
        self,
        initial_cash: float = 1_000_000,
        slippage: float = 0.001,
        fee_rate: float = 0.0003,
    ):
        self.account = Account(initial_cash=initial_cash)
        self.matcher = MatchingEngine(slippage=slippage, fee_rate=fee_rate)
        self.order_counter = 0
        self.fills: List[Fill] = []

    def create_order(self, symbol: str, side: OrderSide, quantity: int,
                      order_type: OrderType = OrderType.MARKET,
                      limit_price: float = None) -> Order:
        """创建订单"""
        self.order_counter += 1
        return Order(
            order_id=self.order_counter,
            symbol=symbol,
            order_type=order_type,
            side=side,
            quantity=quantity,
            limit_price=limit_price,
        )

    def run(
        self,
        price_df: pd.DataFrame,
        signal_fn,  # callable(timestamp, price_df) → List[Order]
    ) -> Dict:
        """运行回测主循环

        Args:
            price_df: 价格数据，必须有列 [open, high, low, close, volume]
            signal_fn: 信号生成函数，返回当天的订单列表

        Returns:
            绩效结果字典
        """
        n = len(price_df)

        for i in range(n):
            row = price_df.iloc[i]
            md = MarketData(
                timestamp=i,
                open=row["open"],
                high=row["high"],
                low=row["low"],
                close=row["close"],
                volume=row.get("volume", 0),
            )

            # 1. 策略生成订单
            orders = signal_fn(i, price_df, self.account)

            # 2. 撮合
            for order in orders:
                fill = self.matcher.match(order, md)
                if fill:
                    # 资金/持仓检查
                    if fill.side == OrderSide.BUY:
                        if not self.account.can_afford(order, fill.price):
                            continue
                    else:
                        if not self.account.has_position(order.symbol, order.quantity):
                            continue

                    self.account.apply_fill(fill, md.close)
                    self.fills.append(fill)

            # 3. 记录快照
            price_map = {"000001": md.close}
            self.account.snapshot(i, price_map)

        # 绩效计算
        result = PerformanceCalculator.calculate(self.account.position_records)
        result["fills"] = self.fills
        result["trades"] = self.account.trade_records
        return result

## 7. 实战：用模拟数据跑完整回测

生成一段模拟价格序列，实现一个简单的均线交叉策略作为信号源。

策略逻辑：
- 短期均线（5日）上穿长期均线（20日）→ 买入信号
- 短期均线（5日）下穿长期均线（20日）→ 卖出信号
- 每次交易 1000 股

In [ ]:
def generate_ohlcv(n=500):
    """生成模拟 OHLCV 数据"""
    np.random.seed(RANDOM_SEED)
    dates = pd.date_range("2020-01-01", periods=n, freq="B")

    # 对数价格 — 前半段上涨，后半段震荡
    mu = np.where(np.arange(n) < n // 3, 0.0008, -0.0002)
    sigma = 0.015
    noise = np.random.randn(n) * sigma
    log_returns = mu + noise

    # 加入一段明显下跌（模拟熊市）
    crash_start = n * 2 // 3
    crash_len = 60
    log_returns[crash_start:crash_start + crash_len] -= 0.003

    price = 100 * np.exp(np.cumsum(log_returns))

    df = pd.DataFrame({
        "date": dates,
        "close": price,
    })

    # 从 close 构造 OHLC
    rets = np.diff(np.log(price), prepend=np.log(price[0]))
    daily_range = np.abs(np.random.randn(n) * 0.01) + 0.005

    df["open"] = df["close"].shift(1).fillna(price[0])
    df["high"] = np.maximum(df["open"], df["close"]) * (1 + daily_range / 2)
    df["low"] = np.minimum(df["open"], df["close"]) * (1 - daily_range / 2)
    df["volume"] = np.random.randint(50000, 200000, n)
    df.set_index("date", inplace=True)

    return df

price_df = generate_ohlcv(500)
print(f"数据范围: {price_df.index[0].date()} ~ {price_df.index[-1].date()}")
print(f"交易日: {len(price_df)}")

# 可视化
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(price_df.index, price_df["close"], linewidth=1, label="close")
axes[0].set_title("模拟价格序列")
axes[0].legend()
axes[1].bar(price_df.index, price_df["volume"], width=1, alpha=0.5)
axes[1].set_title("模拟成交量")
plt.tight_layout()
plt.show()

### 7.1 定义均线交叉策略

In [ ]:
def ma_cross_signal(timestamp: int, price_df: pd.DataFrame, account: Account) -> List[Order]:
    """均线交叉策略信号

    返回当天的订单列表。
    只在交叉信号出现时下单。
    """
    symbol = "000001"
    quantity = 1000  # 固定每次交易 1000 股

    # 至少需要 21 天数据来计算均线
    if timestamp < 21:
        return []

    closes = price_df["close"].values[:timestamp + 1]
    ma_short = np.mean(closes[-5:])
    ma_long = np.mean(closes[-20:])

    # 前一天的均线
    closes_prev = price_df["close"].values[:timestamp]
    if len(closes_prev) >= 20:
        ma_short_prev = np.mean(closes_prev[-5:])
        ma_long_prev = np.mean(closes_prev[-20:])
    else:
        return []

    orders = []

    # 金叉：短均线上穿长均线 → 买入
    if ma_short_prev <= ma_long_prev and ma_short > ma_long:
        close_price = price_df["close"].values[timestamp]
        if account.can_afford(
            Order(0, symbol, OrderType.MARKET, OrderSide.BUY, quantity), close_price
        ):
            orders.append(Order(-1, symbol, OrderType.MARKET, OrderSide.BUY, quantity,
                                timestamp=timestamp))

    # 死叉：短均线下穿长均线 → 卖出
    if ma_short_prev >= ma_long_prev and ma_short < ma_long:
        if account.has_position(symbol, quantity):
            orders.append(Order(-1, symbol, OrderType.MARKET, OrderSide.SELL, quantity,
                                timestamp=timestamp))

    return orders

# 运行回测
engine = BacktestEngine(initial_cash=1_000_000, slippage=0.001, fee_rate=0.0003)
result = engine.run(price_df, signal_fn=ma_cross_signal)

print("=" * 50)
print("均线交叉策略 回测结果")
print("=" * 50)
print(PerformanceCalculator.summary(result))
print(f"  成交笔数: {len(result['fills'])}")

### 7.2 可视化：净值曲线 + 回撤

In [ ]:
nav = result["nav"]
drawdowns = result["drawdowns"]

# 基准（买入持有）
benchmark_nav = 1_000_000 * (price_df["close"].values / price_df["close"].values[0])

# 找出买卖点
trade_dates = []
trade_prices = []
trade_colors = []
for t in result["trades"]:
    idx = t["timestamp"]
    if idx < len(price_df):
        trade_dates.append(price_df.index[idx])
        trade_prices.append(t["price"])
        trade_colors.append("red" if t["side"] == "buy" else "green")

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# 净值
axes[0].plot(price_df.index, nav, label="策略净值", linewidth=1.5, color="steelblue")
axes[0].plot(price_df.index, benchmark_nav, label="买入持有", linewidth=1,
             linestyle="--", alpha=0.6, color="gray")
axes[0].scatter(trade_dates, trade_prices, c=trade_colors, s=20, alpha=0.7, zorder=5)
axes[0].set_title("净值曲线（红点=买入，绿点=卖出）")
axes[0].legend()
axes[0].set_ylabel("净值")

# 回撤
axes[1].fill_between(price_df.index, drawdowns * 100, 0, alpha=0.3, color="coral")
axes[1].plot(price_df.index, drawdowns * 100, linewidth=0.5, color="coral")
axes[1].set_title("回撤 (%)")
axes[1].set_ylabel("回撤")

# 持仓
positions = np.array([r["positions"].get("000001", 0) for r in engine.account.position_records])
axes[2].fill_between(price_df.index, positions, 0, alpha=0.3, color="mediumseagreen")
axes[2].set_title("持仓数量")
axes[2].set_ylabel("股数")
axes[2].set_xlabel("日期")

plt.tight_layout()
plt.show()

# 成交明细
if len(result["trades"]) > 0:
    trades_df = pd.DataFrame(result["trades"])
    print(f"\n成交明细（前10笔）：")
    print(trades_df.head(10).to_string(index=False))

### 7.3 限价单演示

上面用了市价单，现在演示限价单的撮合逻辑。

策略：当价格跌破布林带下轨时，下限价买单（价格再跌 1% 时买入）。

In [ ]:
def limit_order_signal(timestamp: int, price_df: pd.DataFrame, account: Account) -> List[Order]:
    """限价单策略：布林带下轨触发限价买单"""
    symbol = "000001"
    if timestamp < 20:
        return []

    closes = price_df["close"].values[:timestamp + 1]
    ma = np.mean(closes[-20:])
    std = np.std(closes[-20:])
    lower_band = ma - 2 * std
    current_close = price_df["close"].values[timestamp]

    orders = []

    # 价格低于下轨 → 下限价买单（等待更好价格）
    if current_close < lower_band:
        limit_price = current_close * 0.99  # 限价比当前低 1%
        quantity = 500
        if account.can_afford(
            Order(0, symbol, OrderType.LIMIT, OrderSide.BUY, quantity, limit_price=limit_price),
            limit_price
        ):
            orders.append(Order(-1, symbol, OrderType.LIMIT, OrderSide.BUY, quantity,
                                limit_price=limit_price, timestamp=timestamp))

    # 有持仓且价格回到均线上方 → 卖出
    if account.has_position(symbol, 500) and current_close > ma:
        orders.append(Order(-1, symbol, OrderType.MARKET, OrderSide.SELL, 500, timestamp=timestamp))

    return orders

engine2 = BacktestEngine(initial_cash=1_000_000, slippage=0.001, fee_rate=0.0003)
result2 = engine2.run(price_df, signal_fn=limit_order_signal)

print("=" * 50)
print("限价单策略 回测结果")
print("=" * 50)
print(PerformanceCalculator.summary(result2))
print(f"  成交笔数: {len(result2['fills'])}")

# 对比
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(result["nav"], label="市价单策略", linewidth=1.5)
axes[0].plot(result2["nav"], label="限价单策略", linewidth=1.5)
axes[0].plot(benchmark_nav, label="买入持有", linewidth=1, linestyle="--", alpha=0.5)
axes[0].set_title("净值对比")
axes[0].legend()

# 成交价格对比
fill_prices_mkt = [f.price for f in result["fills"] if f.side == OrderSide.BUY]
fill_prices_lmt = [f.price for f in result2["fills"] if f.side == OrderSide.BUY]
if fill_prices_mkt and fill_prices_lmt:
    axes[1].boxplot([fill_prices_mkt, fill_prices_lmt], labels=["市价单", "限价单"])
    axes[1].set_title("买入成交价分布对比")
    axes[1].set_ylabel("成交价")
else:
    axes[1].text(0.5, 0.5, "成交记录不足", ha="center")

plt.tight_layout()
plt.show()

## 8. 手写引擎 vs 成熟框架

手写完回测引擎后，你应该能理解 backtrader / vectorbt / zipline 这些成熟框架的内部原理了。

| 维度 | 手写引擎 | 成熟框架 |
|------|---------|----------|
| **撮合逻辑** | 简化，自己控制 | 支持多种撮合模型、订单簿深度 |
| **手续费/滑点** | 简单比例 | 支持阶梯费率、T+1、涨跌停 |
| **多资产** | 需要自己扩展 | 原生支持 |
| **批量回测** | 需手动循环 | 参数网格搜索、并行计算 |
| **可审计性** | 每行代码都知道做什么 | 框架内部是黑盒 |
| **学习价值** | 极高 | 低（调包即可） |

**建议**：学习阶段用手写引擎理解原理，生产环境用成熟框架保证可靠性。手写过一遍的人，用起 backtrader 来会少很多困惑。

## 9. 小结

这节课我们：

1. **从零构建了一个完整的回测引擎**：订单系统 → 撮合引擎 → 账户管理 → 绩效计算 → 主循环
2. **实现了两种订单类型的撮合逻辑**：市价单立即成交，限价单条件成交
3. **处理了真实回测的摩擦成本**：滑点模拟、手续费扣除、资金/持仓检查
4. **计算了标准绩效指标**：Sharpe、最大回撤、Calmar、胜率、盈亏比
5. **用均线交叉和限价单两个策略做了完整演示**

### 核心收获

回测引擎的本质是 **事件驱动 + 状态管理**。理解了订单从生成到成交的全链路，其余框架都是在这个基础上的功能丰富。

### 下一步

有了手写回测引擎，下一课可以在这个基础上加入风控模块：凯利公式仓位管理、ATR 动态止损、最大回撤限制——这些都是在回测主循环中插入风控检查点即可。

## 验收清单

- [ ] 能画出回测引擎的五大组件及其数据流向
- [ ] 能手写市价单和限价单的撮合逻辑（不查资料）
- [ ] 理解为什么限价买单在 `market_low <= limit_price` 时成交
- [ ] 能自己实现滑点模拟和手续费扣除
- [ ] 能计算 Sharpe / 最大回撤 / Calmar / 胜率 / 盈亏比
- [ ] 理解资金检查和持仓检查在回测中的必要性
- [ ] 跑通了均线交叉 + 限价单两个完整回测